# Data Explorer Agent
### #1 on the DABStep Leaderboard | Built on NVIDIA NeMo Agent Toolkit

---

**Data Explorer** is an AI-powered agent for automated data analysis that achieved state-of-the-art performance on [DABStep](https://huggingface.co/datasets/adyen/DABstep) (Data Agent Benchmark for Multi-step Reasoning), outperforming solutions from AntGroup, Google AI, and the Claude Code baseline.

Built by the **KGMON-LLM Agent Research Team** at NVIDIA:
- Jiwei Liu -- Team Lead, DABStep architecture
- Maximilian Jeblick -- Nemotron Super testing & optimization
- Jack Yu -- EDA workflow & multi-turn chat
- Alessio -- Generic QA agent

## Results

DABStep is 84% "Hard" multi-step reasoning questions over a financial payments dataset. Our approach dominates both accuracy and speed:

| System | Hard Accuracy | Time/Task | Code Length |
|--------|:---:|:---:|:---:|
| **Data Explorer + Haiku 4.5 (Ours)** | **89.95%** | **20s** | **1,870** |
| Data Explorer + Nemotron 3 Super (Ours) | ~77% | 20s | 1,870 |
| DataPilot (AntGroup) | 87.57% | -- | -- |
| Claude Code + Opus 4.5 (baseline) | 66.93% | 10 min | 5,011 |
| DS-STAR (Google AI) | 45.24% | -- | -- |

**30x faster** than the baseline. **+23 points** on hard questions. Less than half the code.

## How It Works: Learn, Infer, Reflect

The key innovation is a **three-phase architecture** that separates expensive learning from fast inference:

```
Phase 1: LEARNING (offline)          Phase 2: INFERENCE (fast)          Phase 3: REFLECTION (offline)
┌──────────────────────┐            ┌──────────────────────┐           ┌──────────────────────┐
│  Heavy Model          │            │  Lightweight Model    │           │  Heavy Model          │
│  (Claude Opus 4.5)    │            │  (Nemotron 3 Super)   │           │  (Claude Sonnet)      │
│                       │   distill  │                       │   review  │                       │
│  Solves training      │──────────▶ │  Uses distilled code  │─────────▶ │  Self-consistency     │
│  tasks, writes        │            │  + few-shot examples  │           │  checks, feeds back   │
│  reusable functions   │            │  to answer in seconds │           │  improved prompts     │
└──────────────────────┘            └──────────────────────┘           └──────────────────────┘
     Output:                              Output:                            Output:
     helper.py (22 functions)             answer + reasoning trace           improved Phase 2 prompts
     new_solutions.md (few-shot)          
```

**Key insight**: By investing upfront in learning and code distillation, even a small, fast model can outsmart heavier models on complex multi-step problems.

This launchable demonstrates **Phase 2 (Inference)** -- the distilled knowledge is already baked in.

## What's Distilled

During the Learning phase, Claude Opus 4.5 solved hundreds of training tasks and distilled the solutions into two artifacts that the inference agent uses:

- **`helper.py`** -- 22 reusable functions for fee matching, transaction filtering, monthly metrics, and edge-case handling
- **`new_solutions.md`** -- few-shot examples showing the agent how to use those functions

Let's see what the agent has to work with:

In [ ]:
import os, inspect, importlib.util
os.chdir('/app')

spec = importlib.util.spec_from_file_location('helper', '/app/dabstep_agent/inference/helper.py')
helper = importlib.util.module_from_spec(spec)
spec.loader.exec_module(helper)

funcs = [(name, getattr(helper, name)) for name, obj in inspect.getmembers(helper)
         if inspect.isfunction(obj) and not name.startswith('_')]

print(f'helper.py: {len(funcs)} distilled functions\n')
for name, fn in funcs:
    sig = inspect.signature(fn)
    doc = (fn.__doc__ or '').strip().split('\n')[0]
    print(f'  {name}{sig}')
    if doc:
        print(f'    -> {doc}')
    print()

## Quick Demo: Ask a Question

The DABStep inference server is running on port 8000. Let's ask it a question about the financial payments dataset (138K transactions, 1000 fee rules, 5 merchants).

In [ ]:
import requests, time
from dotenv import load_dotenv
load_dotenv()

def ask(question, guidelines='N/A'):
    """Send a question to the DABStep inference server."""
    t0 = time.time()
    resp = requests.post('http://localhost:8000/solve',
                         json={'question': question, 'guidelines': guidelines}, timeout=300)
    elapsed = time.time() - t0
    r = resp.json()
    return r['agent_answer'], round(elapsed, 1), r['reasoning_trace']

answer, secs, trace = ask('Which issuing country has the highest number of transactions?')
print(f'Answer:  {answer}')
print(f'Time:    {secs}s')
print(f'Steps:   {len([m for m in trace if m.get("role") == "ai" and m.get("tool_calls")])} tool calls')

In [ ]:
# Try a harder multi-step question
answer, secs, trace = ask(
    'For the 12th of the year 2023, what is the total fees (in euros) '
    'that Belles_cookbook_store should pay?'
)
print(f'Answer:   {answer}')
print(f'Expected: 12.91')
print(f'Time:     {secs}s')

## Try It Yourself

Edit the question below and run the cell. The dataset has info on merchants, card schemes, fees, fraud, and transactions across 2023.

In [ ]:
# ---- Edit your question here ----
my_question = 'What is the average transaction amount by card scheme?'
# ----------------------------------

answer, secs, _ = ask(my_question)
print(f'Answer: {answer}  ({secs}s)')

## Explore the Full Demos

This launchable includes three complete demo notebooks:

| Notebook | What it shows | Model |
|----------|--------------|-------|
| [demo_dabstep.ipynb](demo_dabstep.ipynb) | **DABStep inference** -- the leaderboard-winning agent with before/after comparison showing brute force vs learned pipeline | Nemotron 3 Super |
| [demo_qa.ipynb](demo_qa.ipynb) | **General tabular QA** -- same architecture on any dataset, no prior knowledge or helper functions needed | Claude Haiku 4.5 |
| [demo_eda.ipynb](demo_eda.ipynb) | **Automated EDA** -- agent generates a complete Jupyter notebook with analysis and visualizations from any CSV | GPT-5 mini |

Each demo uses a **different LLM** through NVIDIA Inference Hub, showing that the NeMo Agent Toolkit architecture is model-agnostic.

## Architecture

```
┌────────────────────────────────────────────────────────────┐
│                    NVIDIA Launchable                       │
│                                                            │
│  JupyterLab (port 8888)                                   │
│    ├── START_HERE.ipynb     ← you are here                │
│    ├── demo_dabstep.ipynb   → FastAPI /solve endpoint      │
│    ├── demo_qa.ipynb        → Generic QA (any dataset)     │
│    └── demo_eda.ipynb       → EDA notebook generator       │
│                                                            │
│  DABStep Server (port 8000)                                │
│    └── POST /solve → Tool-calling Agent                    │
│                       ├── helper.py (22 distilled funcs)   │
│                       └── new_solutions.md (few-shot)      │
│                                                            │
│  Data: payments.csv, fees.json, merchant_data.json         │
└────────────────────────────────────────────────────────────┘
          │                    │                    │
          ▼                    ▼                    ▼
   Nemotron 3 Super     Claude Haiku 4.5      GPT-5 mini
   (DABStep agent)      (Generic QA)          (EDA agent)
          └──────────────────┼──────────────────┘
                    NVIDIA Inference Hub
```